# Adel for developers, 2: extend the component

**What you will learn:** add a state variable and a process to `ShootArchitecture` by subclassing it, put the new
component in a plant model, run it, find it in the declarations and the schedule, and test it.

**You need:** developer tutorial 1 (the component formalism). **Run time:** a few seconds.

## 1. A new variable and its process

Suppose a disease model needs each element's **height above the soil** (spores splash up from the ground). A subclass
declares it as a state variable and computes it in a step:

- `state_variable` declares it: its unit, its description, its kind (`descriptor`: a property, not an amount to sum or
  share), its initial value and its scale (`SubOrgan`, the elements);
- the method `_element_height` computes it. Its arguments name the variables it reads (the element's base `z1` and end
  `z2`), the dictionary it returns, the variables it writes. `@postsegmentation` places it in the schedule, after the
  parent's steps, so after the placement;
- `@dataclass` is needed on the subclass, as on every component.

In [1]:
from dataclasses import dataclass

import numpy as np

from openalea.adel.architecture import SCALES, ShootArchitecture
from openalea.metafspm import postsegmentation, state_variable


@dataclass
class ShootWithHeights(ShootArchitecture):
    element_height: float = state_variable(unit="m", description="The height of the element's middle above the soil",
                                           kind="descriptor", default=0., scale=SCALES.SubOrgan)

    @postsegmentation
    def _element_height(self, z1, z2) -> dict:
        return {"element_height": (np.asarray(z1) + np.asarray(z2)) / 2.}

## 2. Where it appears

The new variable is in the declarations, and the new step in the schedule, after the parent's:

In [2]:
declarations = ShootWithHeights.declarations()
print(declarations[declarations["name"] == "element_height"][["name", "kind", "scale", "unit", "description"]]
      .to_string())
print()
for stages, steps in ShootWithHeights.schedule():
    print(f"{', '.join(stages):>18}  {', '.join(steps)}")

              name        kind     scale unit                                        description
91  element_height  descriptor  SubOrgan    m  The height of the element's middle above the soil

      segmentation  _axis_phytomers
  postsegmentation  _elements, _first_inclination, _inclination, _placement, _element_height


## 3. In a plant model

`ShootArchitecture` needs a development model to give it the organs' lengths: the plant model `ScenarioShoot` declares
it with `ScenarioDevelopment` as its parts (coupled by metafspm: each input is the other's variable of the same name,
so no translator is needed), and runs them together in one schedule (`run_together`, in its `grow` step: a call creates every phytomer its thermal time asks for). A plant model with the new component is the same, with the subclass in its place. It is
also the plant's **initiator**, the component that builds each plant's first structure:

In [3]:
from openalea.adel.development import ScenarioDevelopment, ScenarioShoot


class ShootWithHeightsModel(ScenarioShoot):
    initiators = (ShootWithHeights,)
    parts = {"development": ScenarioDevelopment, "architecture": ShootWithHeights}   # shared by name, grown together

## 4. Run it

The users' `simulate` takes a plant model (`model=`); `openalea.adel.development.shoot_scene` builds a scene of it too.
The new variable is on the DataStructure, one value per element:

In [4]:
from openalea.adel.simulation import simulate

grown = simulate(days=60, model=ShootWithHeightsModel)
data_structure = grown.data_structure
height = np.asarray(data_structure.get("element_height"))
print(f"{data_structure.n_nodes()} elements, the highest at {height.max() * 100:.1f} cm")

240 elements, the highest at 21.0 cm


## 5. Test it

A test states what the step must give, on an independent computation. In the package it would go in `test/`, run by
pytest; here it runs in place:

In [5]:
def test_element_height_is_the_middle():
    z1, z2 = np.asarray(data_structure.get("z1")), np.asarray(data_structure.get("z2"))
    np.testing.assert_allclose(height, (z1 + z2) / 2, rtol=0, atol=1e-15)
    assert np.all(height <= np.maximum(z1, z2))


test_element_height_is_the_middle()
print("passed")

passed


## 6. Further

- A variable another model writes is declared with `input_variable(..., by="model_development")`. Named as the
  providing model's variable, it is shared on the DataStructure; under another name (or with a factor, a change of
  scale), a translator links them, the plant model's class attribute `translator` (developer tutorial 1, its coupling
  contract, and metafspm's user guide).
- A process that changes the structure (new phytomers, new elements) belongs in the growth stages (`@segmentation`):
  `ShootArchitecture._axis_phytomers` is the example.
- `example/light_coupling.py` adds a whole component (a toy light model) to the plant model instead of extending one.